# 🏎️ Formula 1 Race Predictions

Predict Formula 1 race results using historical data from **FastF1** and **scikit‑learn** models.

**Workflow:**
1. Select a target race (year + Grand Prix)
2. Automatically collect training data (all prior races that season + historical seasons since 2018)
3. Engineer features → train models → predict the selected race

---

In [ ]:
# Uncomment to install dependencies
# !pip install fastf1 scikit-learn pandas numpy matplotlib seaborn ipywidgets


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import (
    RandomForestRegressor,
    RandomForestClassifier,
    GradientBoostingRegressor,
    GradientBoostingClassifier,
    StackingRegressor,
)
from sklearn.svm import SVC
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.inspection import PartialDependenceDisplay

# Project utilities
from utils.data_loader import (
    setup_cache,
    get_race_schedule,
    get_round_number,
    collect_historical_data,
    collect_season_data,
    load_qualifying_results,
)
from utils.feature_engineering import (
    build_feature_matrix,
    preprocess_features,
    FEATURE_COLS,
    META_COLS,
    TARGET_COL,
)
from utils.evaluation import (
    evaluate_regression,
    evaluate_classification,
    plot_model_comparison,
    plot_feature_importance,
    plot_confusion_matrix,
    plot_prediction_vs_actual,
)

sns.set_theme(style='whitegrid', palette='viridis')
pd.set_option('display.max_columns', 40)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Enable FastF1 disk cache
setup_cache('data/cache')
print('✅ Setup complete')

---
## 2. 🎯 Race Selection

Choose the race you want to predict.  
**Option A** — Set the variables directly in the cell below.  
**Option B** — Use the interactive widgets (requires `ipywidgets`).

In [ ]:
# ── Option A: set variables directly ───────────────────────────
TARGET_YEAR = 2024
TARGET_RACE = 'Monaco'   # Grand Prix name (or part of it)

# ── Option B: interactive widgets (uncomment to use) ─────────
# import ipywidgets as widgets
# from IPython.display import display
#
# year_dd = widgets.Dropdown(
#     options=list(range(2018, 2026)),
#     value=2024,
#     description='Season:',
# )
#
# def on_year_change(change):
#     schedule = get_race_schedule(change['new'])
#     race_dd.options = schedule['EventName'].tolist()
#
# race_dd = widgets.Dropdown(description='Grand Prix:')
# year_dd.observe(on_year_change, names='value')
# on_year_change({'new': year_dd.value})
#
# display(year_dd, race_dd)


In [ ]:
# Resolve round number from the schedule
import difflib

schedule = get_race_schedule(TARGET_YEAR)
event_names = schedule['EventName'].tolist()

# Fuzzy‑match the user input
matches = [n for n in event_names if TARGET_RACE.lower() in n.lower()]
if not matches:
    close = difflib.get_close_matches(TARGET_RACE, event_names, n=3, cutoff=0.4)
    raise ValueError(
        f"'{TARGET_RACE}' not found in {TARGET_YEAR} schedule.\n"
        f"Did you mean one of: {close}?\n"
        f"Available races: {event_names}"
    )

TARGET_EVENT = matches[0]
TARGET_ROUND = int(
    schedule.loc[schedule['EventName'] == TARGET_EVENT, 'RoundNumber'].iloc[0]
)

print(f'🏁 Target race : {TARGET_YEAR} {TARGET_EVENT} (Round {TARGET_ROUND})')
print(f'📊 Training data: 2018–{TARGET_YEAR - 1} (all rounds) + '
      f'{TARGET_YEAR} rounds 1–{TARGET_ROUND - 1}')

# Show season schedule
schedule[['RoundNumber', 'EventName', 'Country', 'EventDate', 'EventFormat']]

---
## 3. 📡 Dynamic Data Collection

Automatically fetch:
- **Historical data** — all races from 2018 to the season before the target year
- **Current‑season data** — all completed rounds *before* the target race
- **Target race qualifying** — used as prediction input

> ⏳ First run may take a while as FastF1 downloads and caches data.

In [ ]:
# Collect historical seasons (2018 → target_year - 1)
HISTORY_START = 2018

if TARGET_YEAR > HISTORY_START:
    print(f'Loading historical data {HISTORY_START}–{TARGET_YEAR - 1}…')
    df_history = collect_historical_data(
        HISTORY_START, TARGET_YEAR - 1,
        progress_callback=lambda msg: print(f'  {msg}')
    )
    print(f'  → {len(df_history)} rows from historical seasons')
else:
    df_history = pd.DataFrame()
    print('No historical seasons before target year.')

In [ ]:
# Collect current‑season races before the target round
if TARGET_ROUND > 1:
    print(f'Loading {TARGET_YEAR} rounds 1–{TARGET_ROUND - 1}…')
    df_current = collect_season_data(
        TARGET_YEAR, TARGET_ROUND,
        progress_callback=lambda msg: print(f'  {msg}')
    )
    print(f'  → {len(df_current)} rows from current season')
else:
    df_current = pd.DataFrame()
    print('⚠️  Round 1 selected — no prior data in this season.')

In [ ]:
# Combine into master training DataFrame
frames = [f for f in [df_history, df_current] if not f.empty]
df_all = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
print(f'\n📦 Combined training set: {len(df_all)} driver‑race rows')
df_all.head()

In [ ]:
# Load target race qualifying (prediction input)
df_target_quali = load_qualifying_results(TARGET_YEAR, TARGET_EVENT)

if df_target_quali is not None:
    print(f'✅ Loaded qualifying for {TARGET_YEAR} {TARGET_EVENT} '
          f'({len(df_target_quali)} drivers)')
else:
    print('⚠️  Qualifying data not available yet. '
          'Predictions will use championship standings as a proxy.')

---
## 4. 🔍 Exploratory Data Analysis

In [ ]:
# Finishing position distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pos = pd.to_numeric(df_all['Position'], errors='coerce').dropna()
axes[0].hist(pos, bins=20, color='#4C72B0', edgecolor='white')
axes[0].set_title('Finishing Position Distribution', fontweight='bold')
axes[0].set_xlabel('Position')
axes[0].set_ylabel('Count')

# Grid vs. Finish
grid = pd.to_numeric(df_all['GridPosition'], errors='coerce')
axes[1].scatter(grid, pos, alpha=0.3, s=20, color='#55A868')
axes[1].plot([1, 20], [1, 20], '--', color='#C44E52', lw=2)
axes[1].set_title('Grid vs. Finish Position', fontweight='bold')
axes[1].set_xlabel('Grid Position')
axes[1].set_ylabel('Finish Position')

plt.tight_layout()
plt.show()

corr = grid.corr(pos)
print(f'Grid↔Finish correlation: {corr:.3f}')

In [ ]:
# Season‑by‑season average finishing position of top teams
top_teams = (
    df_all.groupby('TeamName')['Points']
    .sum()
    .nlargest(5)
    .index.tolist()
)
subset = df_all[df_all['TeamName'].isin(top_teams)].copy()
subset['finish_pos'] = pd.to_numeric(subset['Position'], errors='coerce')

fig, ax = plt.subplots(figsize=(12, 5))
for team in top_teams:
    team_data = subset[subset['TeamName'] == team]
    season_avg = team_data.groupby('Year')['finish_pos'].mean()
    ax.plot(season_avg.index, season_avg.values, marker='o', label=team, linewidth=2)

ax.set_title('Avg. Finishing Position by Season (Top 5 Teams)', fontweight='bold')
ax.set_xlabel('Season')
ax.set_ylabel('Avg. Finish Position')
ax.invert_yaxis()
ax.legend(fontsize=9, loc='best')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

---
## 5. ⚙️ Feature Engineering

Build the feature matrix from raw data. All rolling/cumulative features are computed
using only data available *before* each race (no leakage).

In [ ]:
# Build features
df_features, feature_cols, target_col = build_feature_matrix(df_all)

# Drop rows without a valid target
df_features = df_features.dropna(subset=[target_col]).copy()
print(f'Feature matrix: {df_features.shape[0]} rows × {len(feature_cols)} features')
print(f'Features: {feature_cols}')
df_features.head(10)

In [ ]:
# Correlation heatmap of numeric features
numeric_feats = [c for c in feature_cols if c != 'circuit_type']
corr_matrix = df_features[numeric_feats + [target_col]].corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
    center=0, square=True, ax=ax, linewidths=0.5,
)
ax.set_title('Feature Correlation Heatmap', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Time‑series–aware train / test split
# Train on everything except the last season in our data
last_year = df_features['Year'].max()
last_round = df_features.loc[df_features['Year'] == last_year, 'RoundNumber'].max()

train_mask = ~(
    (df_features['Year'] == last_year) & (df_features['RoundNumber'] == last_round)
)
df_train = df_features[train_mask].copy()
df_test  = df_features[~train_mask].copy()

print(f'Train set : {len(df_train)} rows')
print(f'Test  set : {len(df_test)} rows  (last race in data: {last_year} R{last_round})')

# Preprocess: impute, encode, scale
df_train_pp, scaler, encoder, model_cols = preprocess_features(
    df_train, feature_cols, fit=True
)
df_test_pp, _, _, _ = preprocess_features(
    df_test, feature_cols, fit=False, scaler=scaler, encoder=encoder
)

X_train = df_train_pp[model_cols].values
y_train = df_train_pp[target_col].values
X_test  = df_test_pp[model_cols].values
y_test  = df_test_pp[target_col].values

print(f'\nX_train shape: {X_train.shape}')
print(f'X_test  shape: {X_test.shape}')

---
## 6. 📈 Model Training — Regression

Predict the **finishing position** (1–20) of each driver.

| Model | Why? |
|-|-|
| Ridge | Regularised linear baseline |
| Random Forest | Non‑linear interactions |
| Gradient Boosting | Sequential error‑correction |
| Stacking | Combines all three |

In [ ]:
# Define regression models
ridge = Ridge(alpha=1.0, random_state=RANDOM_STATE)
rf_reg = RandomForestRegressor(
    n_estimators=200, max_depth=10, min_samples_split=5,
    random_state=RANDOM_STATE, n_jobs=-1,
)
gb_reg = GradientBoostingRegressor(
    n_estimators=200, learning_rate=0.1, max_depth=5,
    subsample=0.8, random_state=RANDOM_STATE,
)
stack_reg = StackingRegressor(
    estimators=[
        ('ridge', Ridge(alpha=1.0)),
        ('rf', RandomForestRegressor(n_estimators=100, max_depth=8, random_state=RANDOM_STATE, n_jobs=-1)),
        ('gb', GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=4, random_state=RANDOM_STATE)),
    ],
    final_estimator=Ridge(alpha=0.5),
    n_jobs=-1,
)

reg_models = {
    'Ridge': ridge,
    'Random Forest': rf_reg,
    'Gradient Boosting': gb_reg,
    'Stacking': stack_reg,
}

# Train all regression models
reg_results = []
for name, model in reg_models.items():
    print(f'Training {name}…')
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    metrics = evaluate_regression(y_test, y_pred, model_name=name)
    reg_results.append(metrics)
    print(f'  MAE={metrics["MAE"]:.3f}  RMSE={metrics["RMSE"]:.3f}  R²={metrics["R²"]:.3f}')

pd.DataFrame(reg_results)

In [ ]:
# Regression model comparison
plot_model_comparison(reg_results, 'MAE', title='Regression — Mean Absolute Error (lower is better)')

# Best model scatter plot
best_reg_name = min(reg_results, key=lambda r: r['MAE'])['Model']
best_reg = reg_models[best_reg_name]
y_pred_best = best_reg.predict(X_test)
plot_prediction_vs_actual(y_test, y_pred_best, title=f'{best_reg_name} — Predicted vs Actual')
print(f'🏆 Best regression model: {best_reg_name}')

---
## 7. 🏅 Model Training — Classification

Two binary targets:
- **Podium** — top 3 finish
- **Points** — top 10 finish

In [ ]:
# Create binary targets
y_train_podium = (y_train <= 3).astype(int)
y_test_podium  = (y_test <= 3).astype(int)

y_train_points = (y_train <= 10).astype(int)
y_test_points  = (y_test <= 10).astype(int)

print(f'Podium — train positive rate: {y_train_podium.mean():.2%}')
print(f'Points — train positive rate: {y_train_points.mean():.2%}')

In [ ]:
# Classification models
clf_models = {
    'Logistic Regression': LogisticRegression(
        C=1.0, class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=10, class_weight='balanced',
        random_state=RANDOM_STATE, n_jobs=-1,
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.1, max_depth=5,
        subsample=0.8, random_state=RANDOM_STATE,
    ),
    'SVC': SVC(
        C=1.0, kernel='rbf', probability=True, class_weight='balanced',
        random_state=RANDOM_STATE,
    ),
}

# ── Podium classification ─────────────────────────────────────
print('=== PODIUM (Top 3) ===')
podium_results = []
podium_fitted = {}
for name, model in clf_models.items():
    m = type(model)(**model.get_params())  # fresh copy
    m.fit(X_train, y_train_podium)
    y_pred = m.predict(X_test)
    y_prob = m.predict_proba(X_test)[:, 1] if hasattr(m, 'predict_proba') else None
    metrics = evaluate_classification(y_test_podium, y_pred, y_prob, model_name=name)
    podium_results.append(metrics)
    podium_fitted[name] = m
    print(f'  {name}: F1={metrics["F1 Score"]:.3f}')

print('\n=== POINTS (Top 10) ===')
points_results = []
points_fitted = {}
for name, model in clf_models.items():
    m = type(model)(**model.get_params())
    m.fit(X_train, y_train_points)
    y_pred = m.predict(X_test)
    y_prob = m.predict_proba(X_test)[:, 1] if hasattr(m, 'predict_proba') else None
    metrics = evaluate_classification(y_test_points, y_pred, y_prob, model_name=name)
    points_results.append(metrics)
    points_fitted[name] = m
    print(f'  {name}: F1={metrics["F1 Score"]:.3f}')

In [ ]:
# Classification results tables
print('\n📊 Podium Classification Results')
display(pd.DataFrame(podium_results))

print('\n📊 Points Classification Results')
display(pd.DataFrame(points_results))

---
## 8. 🔬 Model Comparison & Explainability

In [ ]:
# Regression comparison
plot_model_comparison(reg_results, 'MAE', 'Regression — MAE (lower is better)')

# Classification comparisons
plot_model_comparison(podium_results, 'F1 Score', 'Podium — F1 Score (higher is better)')
plot_model_comparison(points_results, 'F1 Score', 'Points — F1 Score (higher is better)')

In [ ]:
# Feature importance from Gradient Boosting regressor
plot_feature_importance(
    reg_models['Gradient Boosting'], model_cols,
    title='Feature Importance (Gradient Boosting Regressor)',
)

# Feature importance from Random Forest classifier (points)
plot_feature_importance(
    points_fitted['Random Forest'], model_cols,
    title='Feature Importance (RF Classifier — Points Finish)',
)

In [ ]:
# Confusion matrix for best podium classifier
best_pod_name = max(podium_results, key=lambda r: r['F1 Score'])['Model']
best_pod = podium_fitted[best_pod_name]
plot_confusion_matrix(
    y_test_podium, best_pod.predict(X_test),
    title=f'Podium — {best_pod_name} Confusion Matrix',
)
print(f'🏆 Best podium classifier: {best_pod_name}')

In [ ]:
# Partial dependence: grid position effect on predicted finish
if 'grid_position' in model_cols:
    fig, ax = plt.subplots(figsize=(8, 5))
    PartialDependenceDisplay.from_estimator(
        reg_models['Gradient Boosting'], X_train,
        features=[model_cols.index('grid_position')],
        feature_names=model_cols, ax=ax,
    )
    ax.set_title('Partial Dependence: Grid Position → Finish Position', fontweight='bold')
    plt.tight_layout()
    plt.show()

---
## 9. 🏁 Predict Selected Race

Use the best‑performing models to predict the target race.

In [ ]:
# Build prediction input from target qualifying + historical context
if df_target_quali is not None and not df_target_quali.empty:
    # Append target quali to the training data so feature engineering
    # can compute rolling stats, then extract just the target rows.
    df_target_quali_copy = df_target_quali.copy()
    # Set a placeholder finish position (NaN — to be predicted)
    df_target_quali_copy['Position'] = np.nan
    df_target_quali_copy['Status'] = 'Predicted'
    df_target_quali_copy['Points'] = 0

    df_for_pred = pd.concat([df_all, df_target_quali_copy], ignore_index=True)
    df_pred_feat, pred_feat_cols, _ = build_feature_matrix(df_for_pred)

    # Extract only the target race rows
    target_mask = (
        (df_pred_feat['Year'] == TARGET_YEAR)
        & (df_pred_feat['RoundNumber'] == TARGET_ROUND)
    )
    df_pred_target = df_pred_feat[target_mask].copy()

    # Preprocess with the fitted scaler/encoder
    df_pred_pp, _, _, _ = preprocess_features(
        df_pred_target, feature_cols, fit=False, scaler=scaler, encoder=encoder
    )
    X_pred = df_pred_pp[model_cols].values
    print(f'Prediction input: {X_pred.shape[0]} drivers')
else:
    print('❌ No qualifying data available — cannot build prediction input.')
    X_pred = None

In [ ]:
# Generate predictions
if X_pred is not None and len(X_pred) > 0:
    pred_position = best_reg.predict(X_pred)

    best_pts_name = max(points_results, key=lambda r: r['F1 Score'])['Model']
    best_pts_clf = points_fitted[best_pts_name]
    pred_points = best_pts_clf.predict(X_pred)
    pred_points_prob = (
        best_pts_clf.predict_proba(X_pred)[:, 1]
        if hasattr(best_pts_clf, 'predict_proba') else pred_points
    )

    pred_podium = best_pod.predict(X_pred)
    pred_podium_prob = (
        best_pod.predict_proba(X_pred)[:, 1]
        if hasattr(best_pod, 'predict_proba') else pred_podium
    )

    # Assemble results table
    results_table = df_pred_target[['Abbreviation', 'TeamName', 'grid_position']].copy()
    results_table['Predicted Position'] = pred_position
    results_table['Podium Prob'] = (pred_podium_prob * 100).round(1)
    results_table['Points Prob'] = (pred_points_prob * 100).round(1)
    results_table = results_table.sort_values('Predicted Position')
    results_table['Predicted Rank'] = range(1, len(results_table) + 1)
    results_table = results_table.reset_index(drop=True)

    print(f'\n🏁 Predicted results for {TARGET_YEAR} {TARGET_EVENT}\n')
    display(
        results_table[[
            'Predicted Rank', 'Abbreviation', 'TeamName',
            'grid_position', 'Predicted Position',
            'Podium Prob', 'Points Prob',
        ]].rename(columns={'grid_position': 'Grid'})
        .style.format({'Predicted Position': '{:.1f}', 'Podium Prob': '{:.1f}%', 'Points Prob': '{:.1f}%'})
        .background_gradient(subset=['Podium Prob'], cmap='Greens')
        .background_gradient(subset=['Points Prob'], cmap='Blues')
    )

In [ ]:
# If the race has already happened, compare with actual results
from utils.data_loader import load_race_results

actual_results = load_race_results(TARGET_YEAR, TARGET_EVENT)

if actual_results is not None and X_pred is not None:
    actual = actual_results[['Abbreviation', 'Position']].copy()
    actual['Actual Position'] = pd.to_numeric(actual['Position'], errors='coerce')

    comparison = results_table.merge(actual[['Abbreviation', 'Actual Position']], on='Abbreviation', how='left')
    comparison['Error'] = (comparison['Predicted Position'] - comparison['Actual Position']).round(1)
    comparison = comparison.sort_values('Actual Position')

    print(f'\n📊 Prediction vs Actual — {TARGET_YEAR} {TARGET_EVENT}\n')
    display(
        comparison[[
            'Abbreviation', 'TeamName', 'Grid',
            'Actual Position', 'Predicted Position', 'Error',
        ]].rename(columns={'grid_position': 'Grid'}).reset_index(drop=True)
    )

    mae = np.abs(comparison['Error'].dropna()).mean()
    print(f'\n🎯 Mean Absolute Error on this race: {mae:.2f} positions')
else:
    print('ℹ️  Race has not been run yet — no actual results to compare.')

---
## 10. 💾 Save Models & Conclusion

In [ ]:
# Save best models
import os
os.makedirs('models', exist_ok=True)

joblib.dump(best_reg, 'models/best_regressor.pkl')
joblib.dump(best_pod, 'models/best_podium_classifier.pkl')
joblib.dump(best_pts_clf, 'models/best_points_classifier.pkl')
joblib.dump(scaler, 'models/scaler.pkl')
joblib.dump(encoder, 'models/encoder.pkl')

print('✅ Models saved to models/ directory')

### Summary

- Collected F1 data from 2018 onwards using **FastF1**
- Engineered 14 features covering driver form, team strength, qualifying pace, weather, and circuit type
- Trained and compared **4 regression** and **4 classification** models using **scikit‑learn**
- Predicted finishing positions, podium, and points finishes for the selected race

### Limitations

- Predictions assume no DNFs, safety cars, or strategic surprises
- Regulation changes between seasons affect model accuracy
- New driver/team combinations have limited historical context

### Next Steps

- Try **XGBoost / LightGBM** for improved gradient boosting performance
- Add **pit‑stop strategy** and **tyre degradation** features
- Build a **Streamlit dashboard** for interactive predictions
- Explore **neural networks** (LSTM) for sequential race‑by‑race modelling